<a href="https://colab.research.google.com/github/16howarthm/pgh-road-ai/blob/main/category_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pittsburgh street-closure category analysis
Reproducible experiment with six balanced categories, fixed prompt-development and held-out splits, and exact 0/1/several-example prompt conditions.

In [10]:
import importlib
import json
import os
import shutil
import subprocess
import sys
import pandas as pd
from pathlib import Path

os.chdir("/content")

REPOSITORY_URL = "https://github.com/16howarthm/pgh-road-ai.git"
REPOSITORY_DIR = Path("/content/pgh-road-ai")

from google.colab import userdata

if REPOSITORY_DIR.exists():
    shutil.rmtree(REPOSITORY_DIR)
subprocess.run(
    [
        "git", "clone", "--branch", "main", "--single-branch",
        REPOSITORY_URL, str(REPOSITORY_DIR),
    ],
    check=True,
)

os.chdir(REPOSITORY_DIR)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        "pandas>=2.0", "scikit-learn>=1.3", "openai>=1.0",
    ],
    check=True,
)

jetstream_api_key = userdata.get("jetstream_api_key")
if not jetstream_api_key:
    raise RuntimeError(
        "jetstream_api_key is unavailable. Add it to Colab Secrets, grant notebook access, "
        "and rerun this cell."
    )
os.environ["jetstream_api_key"] = jetstream_api_key

importlib.invalidate_caches()
sys.modules.pop("road_ai", None)
sys.path.insert(0, str(REPOSITORY_DIR))
road_ai = importlib.import_module("road_ai")
required_road_ai_names = [
    "DEFAULT_CATEGORIES",
    "balanced_sample",
    "load_fixed_splits",
    "make_examples",
    "run_experiment",
    "save_fixed_splits",
    "score_results",
    "split_prompt_test",
    "validate_splits",
]
missing_road_ai_names = [
    name for name in required_road_ai_names if not hasattr(road_ai, name)
]
if missing_road_ai_names:
    raise RuntimeError(
        "Fresh clone road_ai.py is missing required names: "
        + ", ".join(missing_road_ai_names)
    )
from road_ai import (
    DEFAULT_CATEGORIES,
    balanced_sample,
    load_fixed_splits,
    make_examples,
    run_experiment,
    save_fixed_splits,
    score_results,
    split_prompt_test,
    validate_splits,
)

PROJECT_ROOT = REPOSITORY_DIR
DATA_PATH = PROJECT_ROOT / "pgh data.csv"
SPLIT_DIR = PROJECT_ROOT / "category_splits"
OUTPUT_DIR = PROJECT_ROOT / "category_results"
LABEL_COL = "work_type"
CATEGORIES = DEFAULT_CATEGORIES
MODELS = ["llama-4-scout", "gpt-oss-120b"]
PROMPT_CONDITIONS = {"zero": 0, "one": 1, "several": 6}
RANDOM_STATE = 42

print("Project root:", PROJECT_ROOT)
print("Jetstream key available:", bool(jetstream_api_key))

Project root: /content/pgh-road-ai
Jetstream key available: True


## Load or create the fixed splits
The first run de-duplicates permits, reports and removes conflicting-label description groups, samples 50 unique descriptions per category, and saves fixed 120/180 splits. Later runs load those same records.

In [11]:
prompt_path = SPLIT_DIR / "prompt_development.csv"
test_path = SPLIT_DIR / "heldout_test.csv"
audit_path = SPLIT_DIR / "split_audit.json"

if prompt_path.exists() and test_path.exists():
    prompt_dev, test = load_fixed_splits(LABEL_COL, SPLIT_DIR)
    audit = json.loads(audit_path.read_text()) if audit_path.exists() else {}
else:
    df = pd.read_csv(DATA_PATH)
    sample, audit = balanced_sample(
        df, LABEL_COL, CATEGORIES, n_per_category=50,
        random_state=RANDOM_STATE, return_audit=True,
    )
    prompt_dev, test = split_prompt_test(
        sample, LABEL_COL, prompt_n=120, test_n=180, random_state=RANDOM_STATE
    )
    save_fixed_splits(prompt_dev, test, LABEL_COL, SPLIT_DIR, audit)

validate_splits(prompt_dev, test, LABEL_COL)
print(audit)
print("Prompt-development counts:\n", prompt_dev[LABEL_COL].value_counts().sort_index())
print("Held-out counts:\n", test[LABEL_COL].value_counts().sort_index())

{'ambiguous_description_groups': 28, 'ambiguous_records_excluded': 505, 'eligible_permits_before_ambiguity_filter': 12746, 'random_state': 42, 'records_per_category': 50, 'sample_size': 300}
Prompt-development counts:
 work_type
BARRICADE              20
CRANE                  20
DEMOLITION DUMPSTER    20
MACHINERY              20
MATERIALS              20
SCAFFOLD               20
Name: count, dtype: int64
Held-out counts:
 work_type
BARRICADE              30
CRANE                  30
DEMOLITION DUMPSTER    30
MACHINERY              30
MATERIALS              30
SCAFFOLD               30
Name: count, dtype: int64


## Run every model and prompt condition
This fresh run executes all 1,080 requests using the same saved 180-record held-out set. `one` contains exactly one labeled example total; `several` contains six total (one per category).

In [12]:
if not jetstream_api_key:
    raise RuntimeError(
        "jetstream_api_key is unavailable. Add it to Colab Secrets, grant notebook access, "
        "and rerun the setup cell."
    )

all_results = []
for model in MODELS:
    for condition, n_examples in PROMPT_CONDITIONS.items():
        examples = make_examples(
            prompt_dev, LABEL_COL, n_examples=n_examples, random_state=RANDOM_STATE
        )
        assert len(examples) == n_examples
        run = run_experiment(
            test, LABEL_COL, model=model, categories=CATEGORIES, examples=examples
        )
        assert len(run) == 180
        assert run.permit_id.tolist() == test.permit_id.tolist()
        run["model"] = model
        run["prompt_condition"] = condition
        run["n_examples"] = n_examples
        all_results.append(run)

results = pd.concat(all_results, ignore_index=True)
combination_sizes = results.groupby(["model", "prompt_condition"]).size()
assert len(results) == 1080
assert len(combination_sizes) == 6
assert combination_sizes.eq(180).all()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results_path = OUTPUT_DIR / "classification_results.csv"
results.to_csv(results_path, index=False)
assert len(pd.read_csv(results_path)) == 1080
print("Raw results saved to:", results_path)
results.head()

Raw results saved to: /content/pgh-road-ai/category_results/classification_results.csv


,permit_id,description,actual,predicted,confidence,reason,prompt_tokens,completion_tokens,error,model,prompt_condition,n_examples
0,DOMI-CS-2020-01492,ERECT SCAFFOLDING FOR BRICK POINTING.,SCAFFOLD,SCAFFOLD,1.0,The description explicitly mentions erecting s...,136.0,27.0,None,llama-4-scout,zero,0
1,DOMI-CS-2020-01589,FACADE INSPECTIONS USING MAN LIFT,MACHINERY,MACHINERY,1.0,"The description mentions 'MAN LIFT', which is ...",134.0,33.0,None,llama-4-scout,zero,0
2,DOMI-CS-2020-02405,CRANE LIFT AT CMU POSNER HALL - ON TECH ST. - ...,CRANE,CRANE,1.0,The description explicitly mentions a 'CRANE L...,154.0,37.0,None,llama-4-scout,zero,0
3,DOMI-CS-2020-03352,SCAFFOLD FOR MASONRY REPOINTING,SCAFFOLD,SCAFFOLD,1.0,The description explicitly mentions 'SCAFFOLD'...,135.0,40.0,None,llama-4-scout,zero,0
4,DOMI-CS-2020-03505,"DP-2019-03308DEMOLITION OF 209 FIRST AVENUE, P...",BARRICADE,DEMOLITION DUMPSTER,1.0,The description explicitly mentions 'DEMOLITION',151.0,29.0,None,llama-4-scout,zero,0


## Metrics and confusion matrices
Failures and invalid responses remain in the 180-row denominator and appear in the `__FAILED__` confusion-matrix column.

In [13]:
summary_rows = []
matrix_rows = []
for (model, condition), group in results.groupby(["model", "prompt_condition"], sort=False):
    scores = score_results(group, CATEGORIES)
    assert scores["n_scored"] == 180
    summary_rows.append({
        "model": model,
        "prompt_condition": condition,
        "n_examples": int(group.n_examples.iloc[0]),
        "accuracy": scores["accuracy"],
        "macro_f1": scores["macro_f1"],
        "failed_predictions": scores["n_failures"],
        "prompt_tokens": group.prompt_tokens.sum(),
        "completion_tokens": group.completion_tokens.sum(),
    })
    matrix = scores["confusion_matrix"]
    display(model, condition, matrix)
    for actual, row in matrix.iterrows():
        for predicted, count in row.items():
            matrix_rows.append({
                "model": model, "prompt_condition": condition,
                "actual": actual, "predicted": predicted, "count": int(count),
            })

summary = pd.DataFrame(summary_rows)
confusion_matrices = pd.DataFrame(matrix_rows)
summary_path = OUTPUT_DIR / "classification_summary.csv"
confusion_path = OUTPUT_DIR / "confusion_matrices.csv"
summary.to_csv(summary_path, index=False)
confusion_matrices.to_csv(confusion_path, index=False)
assert len(summary) == 6
assert results_path.exists() and summary_path.exists() and confusion_path.exists()
print("All outputs saved under:", OUTPUT_DIR)
display(summary)

'llama-4-scout'

'zero'

,MACHINERY,DEMOLITION DUMPSTER,CRANE,BARRICADE,MATERIALS,SCAFFOLD,__FAILED__
MACHINERY,25,0,0,1,4,0,0
DEMOLITION DUMPSTER,0,27,0,0,2,1,0
CRANE,2,1,19,0,6,1,1
BARRICADE,6,5,1,17,1,0,0
MATERIALS,3,2,0,2,22,1,0
SCAFFOLD,0,0,0,0,3,26,1
__FAILED__,0,0,0,0,0,0,0


'llama-4-scout'

'one'

...

'gpt-oss-120b'

'several'

,MACHINERY,DEMOLITION DUMPSTER,CRANE,BARRICADE,MATERIALS,SCAFFOLD,__FAILED__
MACHINERY,17,0,0,3,10,0,0
DEMOLITION DUMPSTER,0,23,0,0,7,0,0
CRANE,2,0,18,3,6,1,0
BARRICADE,2,6,0,16,6,0,0
MATERIALS,1,0,0,2,27,0,0
SCAFFOLD,0,0,0,0,8,22,0
__FAILED__,0,0,0,0,0,0,0


All outputs saved under: /content/pgh-road-ai/category_results


,model,prompt_condition,n_examples,accuracy,macro_f1,failed_predictions,prompt_tokens,completion_tokens
0,llama-4-scout,zero,0,0.755556,0.759460,2,28817.0,6941.0
1,llama-4-scout,one,1,0.738889,0.736045,1,32261.0,6803.0
2,llama-4-scout,several,6,0.772222,0.774312,1,57137.0,7093.0
3,gpt-oss-120b,zero,0,0.744444,0.747132,0,39805.0,16141.0
4,gpt-oss-120b,one,1,0.705556,0.703503,0,43045.0,17113.0
5,gpt-oss-120b,several,6,0.683333,0.696793,0,68245.0,17489.0


## Error review

In [14]:
errors = results[results.actual.ne(results.predicted)].sort_values(
    ["model", "prompt_condition", "confidence"], na_position="first"
)
display(errors[[
    "model", "prompt_condition", "permit_id", "description",
    "actual", "predicted", "confidence", "reason", "error",
]].head(50))

,model,prompt_condition,permit_id,description,actual,predicted,confidence,reason,error
798,gpt-oss-120b,one,DOMI-CS-2023-04405,CONTRACTOR WILL BE PERFORMING CCTV INSPECTION ...,MACHINERY,BARRICADE,0.30,The description involves a site activity (CCTV...,None
775,gpt-oss-120b,one,DOMI-CS-2022-08484,RESIDENTIAL RENOVATION,MATERIALS,BARRICADE,0.35,Description mentions general renovation withou...,None
886,gpt-oss-120b,one,DOMI-CS-2026-05071,PJ DICK CONSTRUCTION IS CONTINUING RENOVATIONS...,BARRICADE,MACHINERY,0.35,"The description mentions ongoing renovations, ...",None
787,gpt-oss-120b,one,DOMI-CS-2022-17271,DLC WATSON DISTRIBUTION PROJECT WILL BE PERFOR...,MACHINERY,BARRICADE,0.45,The work involves manhole inspections with no ...,None
730,gpt-oss-120b,one,DOMI-CS-2020-06841,"REPLACING HARDSCAPE INFRONT OF HOUSE, WILL NEE...",MATERIALS,MACHINERY,0.70,The description explicitly states the need for...,None
757,gpt-oss-120b,one,DOMI-CS-2021-12554,WELDING OF NEW HAND RAIL ON PROPERTY,MACHINERY,MATERIALS,0.70,"The description involves adding a hand rail, w...",None
771,gpt-oss-120b,one,DOMI-CS-2022-06724,UG PULLING CABLE TO CUSTOMER VAULT,MACHINERY,MATERIALS,0.70,"Description mentions pulling cable, which invo...",None
860,gpt-oss-120b,one,DOMI-CS-2025-08052,DLCO UNDERGROUND TO DO CUT OVERS FOR WATSON SS...,MACHINERY,DEMOLITION DUMPSTER,0.70,"The description mentions 'cut overs', which ar...",None
755,gpt-oss-120b,one,DOMI-CS-2021-11014,SIDE WALK CLOSURE AND HEAVY EQUIPMENT STAGING ...,MACHINERY,CRANE,0.78,The description mentions heavy equipment stagi...,None
792,gpt-oss-120b,one,DOMI-CS-2023-03525,ROOF INSTALLATION,DEMOLITION DUMPSTER,MATERIALS,0.78,Roof installation primarily involves deliverin...,None
